# 🧠 RNN for Sentiment Analysis

In this notebook we build a **Recurrent Neural Network (RNN)** for **sentiment analysis** using PyTorch.

The goal is to classify movie reviews from the **IMDB dataset** as:

- Positive
- Negative

This notebook demonstrates the complete NLP pipeline:

1. Text preprocessing using **Regex**
2. Removing **stopwords**
3. **Stemming** words
4. Converting text to vectors using **TF-IDF**
5. Creating **PyTorch DataLoaders**
6. Building and training an **RNN model**
7. Evaluating model performance

## 📊 Dataset

We use the **IMDB Movie Review dataset**.

Dataset structure:

- `review` → movie review text
- `sentiment` → label (positive / negative)

Example:

**Review:** "This movie was absolutely fantastic!"  
**Sentiment:** positive

**Review:** "The plot was boring and slow."  
**Sentiment:** negative

The dataset is stored in the repository:

**../datasets/imdb_dataset.csv**

## Step 1: Import Required Libraries

In [2]:
# Basic libraries
import pandas as pd
import numpy as np
import re

# NLP preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Machine learning utilities
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Deep learning libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Evaluation
from sklearn.metrics import accuracy_score

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lakba\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Step 2: Load the Dataset

In [5]:
# Load IMDB dataset
df = pd.read_csv("../datasets/imdb_dataset.csv")

# Display first few rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Step 3: Text Preprocessing

We clean the text using several NLP preprocessing steps:

1. Convert text to lowercase  
2. Remove HTML tags and special characters using **Regex**  
3. Remove **stopwords**  
4. Apply **stemming**

This helps reduce noise and improve model performance.

In [6]:
# Initialize preprocessing tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()


def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove non-alphabet characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenize words
    words = text.split()

    # Remove stopwords and apply stemming
    processed_words = []

    for word in words:
        if word not in stop_words:
            processed_words.append(stemmer.stem(word))

    return " ".join(processed_words)


# Apply preprocessing
df['clean_review'] = df['review'].apply(preprocess_text)

df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one review mention watch oz episod hook right ...
1,A wonderful little production. <br /><br />The...,positive,wonder littl product film techniqu unassum old...
2,I thought this was a wonderful way to spend ti...,positive,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,negative,basic famili littl boy jake think zombi closet...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter mattei love time money visual stun film...


## Step 4: Convert Text to Numerical Features (TF-IDF)

Machine learning models cannot process raw text.

Therefore we convert text into **numerical vectors using TF-IDF**.

In [7]:
# Initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_review']).toarray()

# Convert sentiment labels to numeric
y = df['sentiment'].map({'positive':1, 'negative':0}).values

## Step 5: Train-Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

## Step 6: Create PyTorch Dataset

In [9]:
class IMDBDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = IMDBDataset(X_train, y_train)
test_dataset = IMDBDataset(X_test, y_test)

## Step 7: Create DataLoaders

In [10]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

## Step 8: Define RNN Model

In [11]:
class RNNModel(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):

        super(RNNModel, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        # Add sequence dimension
        x = x.unsqueeze(1)

        output, hidden = self.rnn(x)

        out = self.fc(hidden.squeeze(0))

        return out


input_size = X_train.shape[1]
hidden_size = 128
output_size = 2

model = RNNModel(input_size, hidden_size, output_size)

print(model)

RNNModel(
  (rnn): RNN(5000, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


## Step 9: Loss Function and Optimizer

In [12]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Step 10: Train the Model

In [13]:
epochs = 10

for epoch in range(epochs):

    running_loss = 0

    for inputs, labels in train_loader:

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

Epoch [1/10], Loss: 207.8666
Epoch [2/10], Loss: 148.4138
Epoch [3/10], Loss: 139.8420
Epoch [4/10], Loss: 135.5238
Epoch [5/10], Loss: 132.9195
Epoch [6/10], Loss: 131.0274
Epoch [7/10], Loss: 129.4749
Epoch [8/10], Loss: 128.4521
Epoch [9/10], Loss: 127.6988
Epoch [10/10], Loss: 126.5107


## Step 11: Evaluate the Model

In [14]:
model.eval()

predictions = []
actual = []

with torch.no_grad():

    for inputs, labels in test_loader:

        outputs = model(inputs)

        _, predicted = torch.max(outputs, 1)

        predictions.extend(predicted.numpy())
        actual.extend(labels.numpy())

accuracy = accuracy_score(actual, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.8741


## 🎯 Conclusion

In this notebook we implemented a **Recurrent Neural Network for sentiment analysis**.

Pipeline summary:

Raw Text  
→ Regex Cleaning  
→ Stopword Removal  
→ Stemming  
→ TF-IDF Vectorization  
→ PyTorch DataLoader  
→ RNN Model  
→ Sentiment Prediction  


RNNs are powerful models for **sequential data** such as text because they maintain a hidden state that captures